In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.optimize import differential_evolution, minimize, brentq
from scipy.stats import norm
import warnings, json, time
warnings.filterwarnings('ignore')

In [2]:
#loading data
def load_data(path):
    with open(path) as f:
        lines = f.readlines()
    spot = None; data_start = 0
    for i, line in enumerate(lines):
        if 'Last:' in line:
            spot = float(line.split('Last:')[1].split(',')[0].strip())
        if line.startswith('Expiration Date'):
            data_start = i + 1; break
    records = []
    for line in lines[data_start:]:
        p = [x.strip() for x in line.split(',')]
        if len(p) < 21: continue
        try:
            records.append(dict(
                exp_date=p[0], strike=float(p[11]),
                call_bid=float(p[4]), call_ask=float(p[5]),
                put_bid=float(p[15]),  put_ask=float(p[16]),
                call_iv=float(p[7]),   put_iv=float(p[17]),
                call_mid=(float(p[4])+float(p[5]))/2,
                put_mid=(float(p[15])+float(p[16]))/2,
                call_spread=float(p[5])-float(p[4]),
                put_spread=float(p[16])-float(p[15]),))
        except: continue
    df = pd.DataFrame(records)
    df['exp_date'] = pd.to_datetime(df['exp_date'], format='%a %b %d %Y')
    df['T'] = (df['exp_date'] - pd.Timestamp('2026-04-12')).dt.days / 365.0
    return spot, df

data = "/Users/pparsi27/Downloads/spx_quotedata.csv"
SPOT, df_all = load_data(data)
R = 0.053    # SOFR / Fed Funds, April 2026

In [3]:
#build option slices
def estimate_q(grp, T, r, S):
    """Put-call parity at ATM strike to estimate dividend yield."""
    atm = grp.assign(d=abs(grp['strike']-S)).nsmallest(1,'d').iloc[0]
    K, C, P = atm['strike'], atm['call_mid'], atm['put_mid']
    val = (C - P + K*np.exp(-r*T)) / S
    if val <= 0: return 0.015
    return float(np.clip(-np.log(val)/T, 0, 0.04))

slice_info = []
for exp, grp in df_all.groupby('exp_date'):
    T = grp['T'].iloc[0]
    if T < 0.003: continue
    q = estimate_q(grp, T, R, SPOT)
    calls = grp[grp['strike'] >= SPOT][['strike','call_mid','call_spread','call_iv']].copy()
    calls.columns = ['K','mid','spread','iv']; calls['type'] = 'C'
    puts  = grp[grp['strike'] <= SPOT][['strike','put_mid','put_spread','put_iv']].copy()
    puts.columns  = ['K','mid','spread','iv']; puts['type']  = 'P'
    otm = pd.concat([puts, calls]).drop_duplicates('K').sort_values('K')
    otm = otm[otm['mid'] > 0.5].reset_index(drop=True)
    if len(otm) < 3: continue
    slice_info.append({'exp':exp, 'T':T, 'q':q, 'otm':otm,
                       'K': otm['K'].values.astype(float),
                       'mkt': otm['mid'].values.astype(float),
                       'sp': otm['spread'].values.astype(float),
                       'tp': otm['type'].values})

Q_AVG = float(np.mean([s['q'] for s in slice_info]))
print(f"Spot={SPOT:.2f}  Rate={R:.4f}  Q_avg={Q_AVG:.4f}  Slices={len(slice_info)}")

Spot=6816.89  Rate=0.0530  Q_avg=0.0400  Slices=13


In [4]:
# option pricing functions
# Black-Scholes (for IV inversion)
def bs_price(S, K, T, r, q, sig, flag='C'):
    if T <= 0 or sig <= 1e-6:
        return max((S*np.exp(-q*T)-K*np.exp(-r*T))*(1 if flag=='C' else -1), 0.)
    d1 = (np.log(S/K)+(r-q+.5*sig**2)*T)/(sig*np.sqrt(T))
    d2 = d1 - sig*np.sqrt(T)
    if flag == 'C': return S*np.exp(-q*T)*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)
    return K*np.exp(-r*T)*norm.cdf(-d2) - S*np.exp(-q*T)*norm.cdf(-d1)

def implied_vol(price, S, K, T, r, q, flag):
    intr = max((S*np.exp(-q*T)-K*np.exp(-r*T))*(1 if flag=='C' else -1), 0.)
    if price <= intr + 1e-6: return np.nan
    try: return brentq(lambda v: bs_price(S,K,T,r,q,v,flag)-price, 1e-4, 5., xtol=1e-6)
    except: return np.nan

# Heston
def heston_calls(params, S, K_arr, T, r, q):
    """
    Vectorised Heston call pricer via two-measure Gil-Pelaez inversion.
    Uses Gatheral's C(u) formulation with the explicit P1/P2 decomposition.
    """
    kappa, theta, sigma, rho, v0 = params
    NFFT = 2048; dphi = 0.05
    u_vec = np.arange(1, NFFT+1, dtype=float) * dphi

    def _hcf(u, j):
        """Characteristic function for measure j (j=1: asset, j=2: risk-neutral)."""
        u = u + 0j
        if j == 1: uj, bj = 0.5, kappa - rho*sigma
        else:      uj, bj = -0.5, kappa
        d = np.sqrt((rho*sigma*1j*u - bj)**2 - sigma**2*(2*uj*1j*u - u**2) + 0j)
        g = (bj - rho*sigma*1j*u + d) / (bj - rho*sigma*1j*u - d)
        edT = np.exp(d*T)
        C = (r-q)*1j*u*T + (kappa*theta/sigma**2)*((bj-rho*sigma*1j*u+d)*T
            - 2*np.log((1-g*edT)/(1-g)))
        D = (bj - rho*sigma*1j*u + d)/sigma**2 * (1-edT)/(1-g*edT)
        return np.exp(C + D*v0)

    phi1 = _hcf(u_vec, 1)  # (NFFT,)
    phi2 = _hcf(u_vec, 2)

    # log-moneyness: x = ln(S/K) for each K
    x = np.log(S / K_arr)          # (nK,)
    u_ = u_vec[:, None]             # (NFFT,1)
    exp_ = np.exp(1j * u_ * x)     # (NFFT,nK)

    integ1 = np.real(exp_ * phi1[:, None] / (1j * u_)).sum(0)
    integ2 = np.real(exp_ * phi2[:, None] / (1j * u_)).sum(0)
    P1 = np.clip(0.5 + dphi/np.pi * integ1, 0, 1)
    P2 = np.clip(0.5 + dphi/np.pi * integ2, 0, 1)
    return np.maximum(S*np.exp(-q*T)*P1 - K_arr*np.exp(-r*T)*P2, 0.0)

# VGSA 
def _cir_phi(u, T, y0, kappa, eta, lam):
    """CIR time-change characteristic function E[exp(iu*Y(T))]."""
    gam = np.sqrt(kappa**2 - 2*lam**2*1j*u + 0j)
    sinh_ = np.sinh(gam*T/2); cosh_ = np.cosh(gam*T/2); tanh_ = np.tanh(gam*T/2)
    A = np.exp(kappa**2*eta*T/lam**2) / (cosh_ + (kappa/gam)*sinh_)**(2*kappa*eta/lam**2)
    B = (2*1j*u) / (kappa + gam/tanh_)
    return A * np.exp(B * y0)

def vgsa_calls(params, S, K_arr, T, r, q):
    """
    VGSA call pricer.  Martingale correction applied via omega = -ln(E[e^Z])/T.
    Integrand uses x = ln(S/K) + (r-q+omega)*T (log forward moneyness).
    """
    sigma, nu, theta_v, kappa, eta, lam = params
    y0 = 1.0 / nu

    def vg_psi(u):
        return -(1/nu) * np.log(1 - 1j*u*theta_v*nu + sigma**2*nu/2 * u**2 + 0j)

    def phi_Z(u):
        return _cir_phi(-1j*vg_psi(u), T, y0, kappa, eta, lam)

    # Martingale correction
    phi_mi = float(np.real(phi_Z(np.array([-1j + 0j]))[0]))
    if phi_mi <= 0: return np.zeros(len(K_arr))
    omega = -np.log(phi_mi) / T

    NFFT = 2048; dphi = 0.05
    u_vec = np.arange(1, NFFT+1, dtype=float) * dphi

    pz  = phi_Z(u_vec + 0j)        # phi_Z(u)
    pzm = phi_Z(u_vec - 1j)        # phi_Z(u-i)

    # log forward moneyness for each K
    x = np.log(S / K_arr) + (r - q + omega) * T   # (nK,)
    u_ = u_vec[:, None]
    exp_ = np.exp(1j * u_ * x)

    integ2 = np.real(exp_ * pz[:, None]  / (1j * u_)).sum(0)
    integ1 = np.real(exp_ * pzm[:, None] / (1j * u_ * phi_mi)).sum(0)
    P1 = np.clip(0.5 + dphi/np.pi * integ1, 0, 1)
    P2 = np.clip(0.5 + dphi/np.pi * integ2, 0, 1)
    return np.maximum(S*np.exp(-q*T)*P1 - K_arr*np.exp(-r*T)*P2, 0.0)

def otm_prices(calls, K_arr, S, T, r, q, tp_arr):
    """Convert call prices to put prices via put-call parity where needed."""
    puts = np.maximum(calls - S*np.exp(-q*T) + K_arr*np.exp(-r*T), 0.)
    return np.where(tp_arr == 'C', calls, puts)


In [5]:
# calibration functions
def make_obj(model_fn, wt_mode):
    def obj(x):
        tot = 0.0
        for s in slice_info:
            T, q = s['T'], s['q']
            Ka, mkt, sp, tp = s['K'], s['mkt'], s['sp'], s['tp']
            try:
                calls = model_fn(x, SPOT, Ka, T, R, q)
                mdl   = otm_prices(calls, Ka, SPOT, T, R, q, tp)
            except: return 1e15
            err = mdl - mkt
            w = np.ones(len(Ka)) if wt_mode == 'equal' else 1.0/np.maximum(sp, 0.5)
            tot += (w * err**2).sum()
        return tot
    return obj

In [6]:
#error analysis functions
def error_table(p_eq, p_sw, model_fn):
    rows = []
    for s in slice_info:
        T, q = s['T'], s['q']
        Ka, mkt, sp, tp = s['K'], s['mkt'], s['sp'], s['tp']
        try:
            c_eq = otm_prices(model_fn(p_eq,SPOT,Ka,T,R,q), Ka,SPOT,T,R,q,tp)
            c_sw = otm_prices(model_fn(p_sw,SPOT,Ka,T,R,q), Ka,SPOT,T,R,q,tp)
        except: continue
        for i, (K, mkt_p, spr, fl) in enumerate(zip(Ka, mkt, sp, tp)):
            iv_mkt = s['otm'].iloc[i]['iv']
            iv_eq  = implied_vol(c_eq[i], SPOT, K, T, R, q, fl)
            iv_sw  = implied_vol(c_sw[i], SPOT, K, T, R, q, fl)
            rows.append({'T':T, 'K':K, 'flag':fl, 'mkt_iv':iv_mkt,
                         'iv_eq':iv_eq, 'iv_sw':iv_sw, 'spread':spr})
    df = pd.DataFrame(rows)
    df['err_eq'] = df['iv_eq'] - df['mkt_iv']
    df['err_sw'] = df['iv_sw'] - df['mkt_iv']
    return df

def stats(df, col):
    v = df[col].dropna()
    return {'RMSE': float(np.sqrt((v**2).mean())),
            'MAE':  float(v.abs().mean()),
            'Bias': float(v.mean()),
            'MaxErr': float(v.abs().max())}

In [7]:
#call prices and local volatility functions
def build_surface(params, model_fn):
    surf = np.zeros((len(T_grid), len(K_grid)))
    for i, T in enumerate(T_grid):
        try:
            c = model_fn(params, SPOT, K_grid, T, R, Q_AVG)
            surf[i] = np.maximum(c, 0.)
        except:
            surf[i] = np.maximum(SPOT*np.exp(-Q_AVG*T) - K_grid*np.exp(-R*T), 0.)
    return surf

def dupire_lv(C, K_arr, T_arr, r, q):
    """Dupire (1994) local volatility: σ²(K,T) = [∂C/∂T + qC + (r-q)K∂C/∂K] / [½K²∂²C/∂K²]"""
    dK = K_arr[1] - K_arr[0]; dT = T_arr[1] - T_arr[0]
    lv = np.full_like(C, np.nan)
    for i in range(1, len(T_arr)-1):
        for j in range(1, len(K_arr)-1):
            c = C[i,j]
            dC_dT   = (C[i+1,j] - C[i-1,j]) / (2*dT)
            dC_dK   = (C[i,j+1] - C[i,j-1]) / (2*dK)
            d2C_dK2 = (C[i,j+1] - 2*c + C[i,j-1]) / dK**2
            num = dC_dT + q*c + (r-q)*K_arr[j]*dC_dK
            den = 0.5 * K_arr[j]**2 * d2C_dK2
            if den > 1e-10 and num > 0:
                lv[i,j] = np.sqrt(num/den)
    return np.clip(lv, 0.01, 1.0)

In [8]:
BOUNDS_H = [(0.1,10),(0.005,0.5),(0.05,1.4),(-0.99,-0.01),(0.001,0.5)]
BOUNDS_V  = [(0.05,0.8),(0.1,5.0),(-0.5,0.2),(0.1,8),(0.1,3),(0.05,1.5)]

DE_OPTS = dict(seed=42, maxiter=200, tol=1e-5, mutation=(0.5,1),
               recombination=0.9, popsize=10, workers=1, polish=True, disp=False)

print("Calibrating Heston (equal weights)")
t0 = time.time()
res = differential_evolution(make_obj(heston_calls, 'equal'), BOUNDS_H, **DE_OPTS)
heston_eq = res.x
print(f"  κ={heston_eq[0]:.4f}  θ={heston_eq[1]:.4f}  σ={heston_eq[2]:.4f}  "
      f"ρ={heston_eq[3]:.4f}  v₀={heston_eq[4]:.4f}  loss={res.fun:.2f}  "
      f"t={time.time()-t0:.1f}s")

print("Calibrating Heston (spread-weighted)")
t0 = time.time()
res = differential_evolution(make_obj(heston_calls, 'spread'), BOUNDS_H, **DE_OPTS)
heston_sw = res.x
print(f"  κ={heston_sw[0]:.4f}  θ={heston_sw[1]:.4f}  σ={heston_sw[2]:.4f}  "
      f"ρ={heston_sw[3]:.4f}  v₀={heston_sw[4]:.4f}  loss={res.fun:.2f}  "
      f"t={time.time()-t0:.1f}s")

print("Calibrating VGSA (equal weights)")
t0 = time.time()
res = differential_evolution(make_obj(vgsa_calls, 'equal'), BOUNDS_V, **DE_OPTS)
vgsa_eq = res.x
print(f"  σ={vgsa_eq[0]:.4f}  ν={vgsa_eq[1]:.4f}  θ={vgsa_eq[2]:.4f}  "
      f"κ={vgsa_eq[3]:.4f}  η={vgsa_eq[4]:.4f}  λ={vgsa_eq[5]:.4f}  "
      f"loss={res.fun:.2f}  t={time.time()-t0:.1f}s")

print("Calibrating VGSA (spread-weighted)")
t0 = time.time()
res = differential_evolution(make_obj(vgsa_calls, 'spread'), BOUNDS_V, **DE_OPTS)
vgsa_sw = res.x
print(f"  σ={vgsa_sw[0]:.4f}  ν={vgsa_sw[1]:.4f}  θ={vgsa_sw[2]:.4f}  "
      f"κ={vgsa_sw[3]:.4f}  η={vgsa_sw[4]:.4f}  λ={vgsa_sw[5]:.4f}  "
      f"loss={res.fun:.2f}  t={time.time()-t0:.1f}s")

print("Computing IV error tables")
h_err = error_table(heston_eq, heston_sw, heston_calls)
v_err = error_table(vgsa_eq,   vgsa_sw,   vgsa_calls)
h_eq_st=stats(h_err,'err_eq'); h_sw_st=stats(h_err,'err_sw')
v_eq_st=stats(v_err,'err_eq'); v_sw_st=stats(v_err,'err_sw')

print("IV ERROR SUMMARY")
for nm, st in [("Heston EW",h_eq_st),("Heston SW",h_sw_st),("VGSA EW",v_eq_st),("VGSA SW",v_sw_st)]:
    print(f"  {nm}: RMSE={st['RMSE']:.4f}  MAE={st['MAE']:.4f}  Bias={st['Bias']:.4f}  Max={st['MaxErr']:.4f}")

K_grid = np.linspace(0.78*SPOT, 1.22*SPOT, 50)
T_grid = np.linspace(3/365, 0.14, 35)

print("Building call surfaces")
H_eq=build_surface(heston_eq,heston_calls); H_sw=build_surface(heston_sw,heston_calls)
V_eq=build_surface(vgsa_eq,  vgsa_calls);   V_sw=build_surface(vgsa_sw,  vgsa_calls)

print("Computing local volatility surfaces (Dupire)")
LVheq=dupire_lv(H_eq,K_grid,T_grid,R,Q_AVG); LVhsw=dupire_lv(H_sw,K_grid,T_grid,R,Q_AVG)
LVveq=dupire_lv(V_eq,K_grid,T_grid,R,Q_AVG); LVvsw=dupire_lv(V_sw,K_grid,T_grid,R,Q_AVG)



=== Calibrating Heston (equal weights) ===
  κ=4.5149  θ=0.0050  σ=1.4000  ρ=-0.9900  v₀=0.0352  loss=37960.16  t=138.5s
=== Calibrating Heston (spread-weighted) ===
  κ=0.1000  θ=0.0050  σ=1.4000  ρ=-0.9900  v₀=0.0318  loss=25927.67  t=112.7s
=== Calibrating VGSA (equal weights) ===
  σ=0.0500  ν=0.1000  θ=-0.1120  κ=8.0000  η=0.1000  λ=0.0686  loss=43003.08  t=283.1s
=== Calibrating VGSA (spread-weighted) ===
  σ=0.0500  ν=0.1066  θ=-0.1130  κ=8.0000  η=0.1000  λ=0.0713  loss=28300.30  t=246.7s

Computing IV error tables...

=== IV ERROR SUMMARY ===
  Heston EW: RMSE=219.4526  MAE=81.5895  Bias=-81.5734  Max=1169.8245
  Heston SW: RMSE=219.4559  MAE=81.5941  Bias=-81.5804  Max=1169.8337
  VGSA EW: RMSE=219.4575  MAE=81.5957  Bias=-81.5768  Max=1169.8340
  VGSA SW: RMSE=219.4596  MAE=81.5974  Bias=-81.5814  Max=1169.8399

Building call surfaces...
Computing local volatility surfaces (Dupire)...


In [16]:
CH='#2166ac'; CV='#d6604d'
plt.rcParams.update({'font.size':9,'axes.titlesize':10})

# Fig 1 – Market smiles
fig, axes = plt.subplots(2, 4, figsize=(15,7))
axes = axes.flatten()
for idx, s in enumerate(slice_info[:8]):
    ax = axes[idx]; m = s['otm']['K']/SPOT
    ax.plot(m, s['otm']['iv'], 'ko-', ms=3, lw=1.5)
    ax.set_title(f"{s['exp'].strftime('%b %d')}  T={s['T']:.3f}y", fontsize=8)
    ax.set_xlabel('K/S'); ax.set_ylabel('IV'); ax.grid(alpha=0.25); ax.set_ylim(0.10,0.30)
fig.suptitle('Market Implied Volatility Smiles — SPX OTM Options, April 2026', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
plt.savefig('/Users/pparsi27/Downloads/fig1.png',dpi=120,bbox_inches='tight')

# Fig 2 – 3D Call surfaces
fig = plt.figure(figsize=(14,5)); KK,TT=np.meshgrid(K_grid/SPOT, T_grid)
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(KK,TT,H_eq,cmap='Blues',alpha=0.85)
ax1.set(xlabel='K/S',ylabel='T(yr)',zlabel='Call $',
        title=f'Heston (EW)  κ={heston_eq[0]:.2f} θ={heston_eq[1]:.3f}\nσ={heston_eq[2]:.2f} ρ={heston_eq[3]:.2f} v₀={heston_eq[4]:.4f}')
ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_surface(KK,TT,V_eq,cmap='Reds',alpha=0.85)
ax2.set(xlabel='K/S',ylabel='T(yr)',zlabel='Call $',
        title=f'VGSA (EW)  σ={vgsa_eq[0]:.2f} ν={vgsa_eq[1]:.2f} θ={vgsa_eq[2]:.2f}\nκ={vgsa_eq[3]:.2f} η={vgsa_eq[4]:.2f} λ={vgsa_eq[5]:.2f}')
fig.suptitle('Model Call Price Surfaces', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
plt.savefig('/Users/pparsi27/Downloads/fig2.png',dpi=120,bbox_inches='tight')

# Fig 3 – Local vol contour maps
fig, axes = plt.subplots(2,2,figsize=(13,9))
Ki=K_grid[1:-1]/SPOT; Ti=T_grid[1:-1]
for ax,data,title,cm in zip(axes.flatten(),
    [LVheq,LVhsw,LVveq,LVvsw],
    ['Heston EW','Heston SW','VGSA EW','VGSA SW'],
    ['Blues_r','Blues','Reds_r','Reds']):
    im=ax.contourf(Ki,Ti,data[1:-1,1:-1],levels=20,cmap=cm)
    plt.colorbar(im,ax=ax,label='σ_loc')
    CS=ax.contour(Ki,Ti,data[1:-1,1:-1],levels=8,colors='k',lw=0.4,alpha=0.3)
    ax.clabel(CS,inline=True,fontsize=6)
    ax.set(xlabel='K/S',ylabel='T (years)',title=f'Local Vol — {title}')
fig.suptitle('Local Volatility Surfaces σ_loc(K,T)  [Dupire Formula]',fontsize=13,fontweight='bold')
plt.tight_layout()
plt.savefig('/Users/pparsi27/Downloads/fig3.png',dpi=120,bbox_inches='tight')

# Fig 4 – Cross-sections
fig, axes = plt.subplots(1,2,figsize=(13,5))
Ti_idx=len(T_grid)//2; Ti_val=T_grid[Ti_idx]; Ki_idx=np.argmin(np.abs(K_grid-SPOT))
ax=axes[0]
for d,lbl,c,ls in [(LVheq,'Heston EW',CH,'-'),(LVhsw,'Heston SW',CH,'--'),
                    (LVveq,'VGSA EW',CV,'-'),(LVvsw,'VGSA SW',CV,'--')]:
    ax.plot(Ki,d[Ti_idx,1:-1],color=c,ls=ls,lw=2,label=lbl)
ax.set(xlabel='K/S',ylabel='Local Vol',title=f'Strike Smile at T={Ti_val:.2f}yr'); ax.legend(); ax.grid(alpha=0.3)
ax=axes[1]
for d,lbl,c,ls in [(LVheq,'Heston EW',CH,'-'),(LVhsw,'Heston SW',CH,'--'),
                    (LVveq,'VGSA EW',CV,'-'),(LVvsw,'VGSA SW',CV,'--')]:
    ax.plot(Ti,d[1:-1,Ki_idx],color=c,ls=ls,lw=2,label=lbl)
ax.set(xlabel='T (years)',ylabel='Local Vol',title='ATM Term Structure'); ax.legend(); ax.grid(alpha=0.3)
fig.suptitle('Local Volatility Cross-Sections',fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig('/Users/pparsi27/Downloads/fig4.png',dpi=120,bbox_inches='tight')

# Fig 5 – IV error scatter
fig,axes=plt.subplots(2,2,figsize=(13,9))
for ax,(df,col,title,clr) in zip(axes.flatten(),[
    (h_err,'err_eq','Heston Equal Weights',CH),
    (h_err,'err_sw','Heston Spread-Weighted','#084081'),
    (v_err,'err_eq','VGSA Equal Weights',CV),
    (v_err,'err_sw','VGSA Spread-Weighted','#a50026')]):
    v=df.dropna(subset=[col])
    ax.scatter(v['K']/SPOT,v[col]*100,alpha=0.6,s=18,c=clr,edgecolors='none')
    ax.axhline(0,color='k',lw=1)
    ax.set(xlabel='K/S',ylabel='IV Error (vol-pts ×100)',title=title); ax.grid(alpha=0.25)
    s_=stats(df,col)
    ax.text(0.03,0.93,f"RMSE={s_['RMSE']*100:.2f}  Bias={s_['Bias']*100:.2f} vpts",
            transform=ax.transAxes,fontsize=8.5,bbox=dict(boxstyle='round',fc='w',alpha=0.7))
fig.suptitle('Implied Volatility Error: Model vs Market',fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig('/Users/pparsi27/Downloads/fig5.png',dpi=120,bbox_inches='tight')

# Fig 6 – IV smile fit for selected slice
s=slice_info[min(4,len(slice_info)-1)]
T,q=s['T'],s['q']; Ka,tpa,mktiv=s['K'],s['tp'],s['otm']['iv'].values
def get_iv(params,fn):
    try:
        c=otm_prices(fn(params,SPOT,Ka,T,R,q),Ka,SPOT,T,R,q,tpa)
        return np.array([implied_vol(c[i],SPOT,K,T,R,q,fl) for i,(K,fl) in enumerate(zip(Ka,tpa))])
    except: return np.full(len(Ka),np.nan)

fig,ax=plt.subplots(figsize=(10,5)); m=Ka/SPOT
ax.plot(m,mktiv,'ko-',lw=2,ms=6,label='Market IV',zorder=5)
ax.plot(m,get_iv(heston_eq,heston_calls),color=CH,lw=2,label='Heston EW')
ax.plot(m,get_iv(heston_sw,heston_calls),color=CH,lw=2,ls='--',label='Heston SW')
ax.plot(m,get_iv(vgsa_eq,  vgsa_calls  ),color=CV,lw=2,label='VGSA EW')
ax.plot(m,get_iv(vgsa_sw,  vgsa_calls  ),color=CV,lw=2,ls='--',label='VGSA SW')
ax.axvline(1,color='gray',ls=':',lw=1)
ax.set(xlabel='K/S',ylabel='Implied Vol',title=f'IV Smile Fit — {s["exp"].strftime("%b %d %Y")} (T={T:.4f}yr)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/Users/pparsi27/Downloads/fig6.png',dpi=120,bbox_inches='tight')

# Fig 7 – 3D local vol comparison
fig=plt.figure(figsize=(13,5)); KK2,TT2=np.meshgrid(Ki,Ti)
ax1=fig.add_subplot(121,projection='3d')
ax1.plot_surface(KK2,TT2,LVheq[1:-1,1:-1],cmap='Blues_r',alpha=0.85)
ax1.set(xlabel='K/S',ylabel='T',zlabel='σ_loc',title='Heston Local Vol (EW)')
ax2=fig.add_subplot(122,projection='3d')
ax2.plot_surface(KK2,TT2,LVveq[1:-1,1:-1],cmap='Reds_r',alpha=0.85)
ax2.set(xlabel='K/S',ylabel='T',zlabel='σ_loc',title='VGSA Local Vol (EW)')
fig.suptitle('Local Volatility 3D: Heston vs VGSA',fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig('/Users/pparsi27/Downloads/fig7.png',dpi=120,bbox_inches='tight')

# Fig 8 – EW vs SW parameter sensitivity
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax, (df_e,col_eq,col_sw,title) in zip(axes,[
    (h_err,'err_eq','err_sw','Heston: EW vs SW IV Errors'),
    (v_err,'err_eq','err_sw','VGSA: EW vs SW IV Errors')]):
    v_eq=df_e['err_eq'].dropna()*100; v_sw=df_e['err_sw'].dropna()*100
    ax.boxplot([v_eq,v_sw],labels=['Equal Wt','Spread Wt'],patch_artist=True,
               boxprops=dict(facecolor='lightblue',color='navy'),
               medianprops=dict(color='red',lw=2))
    ax.axhline(0,color='k',lw=1,ls='--')
    ax.set(ylabel='IV Error (vol-pts)',title=title); ax.grid(alpha=0.3)
fig.suptitle('Weighting Scheme Comparison: IV Error Distribution',fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig('/Users/pparsi27/Downloads/fig8.png',dpi=120,bbox_inches='tight')


In [18]:
results = {
    'spot':SPOT, 'rate':R, 'q_avg':Q_AVG,
    'heston_eq':list(heston_eq), 'heston_sw':list(heston_sw),
    'vgsa_eq':list(vgsa_eq), 'vgsa_sw':list(vgsa_sw),
    'heston_stats_eq':h_eq_st, 'heston_stats_sw':h_sw_st,
    'vgsa_stats_eq':v_eq_st, 'vgsa_stats_sw':v_sw_st,
}
print(results)

{'spot': 6816.8901, 'rate': 0.053, 'q_avg': 0.039999999999999994, 'heston_eq': [4.514937752269386, 0.005001754459037573, 1.3999960278438268, -0.9899999097325554, 0.03515897534012631], 'heston_sw': [0.1, 0.005, 1.4, -0.99, 0.031811065415303444], 'vgsa_eq': [0.05, 0.1, -0.1119903984437546, 8.0, 0.1, 0.06856616762892953], 'vgsa_sw': [0.05, 0.10663014283767179, -0.1129968032451567, 8.0, 0.1, 0.07130640077636427], 'heston_stats_eq': {'RMSE': 219.45256368538304, 'MAE': 81.58948930303258, 'Bias': -81.57335857697969, 'MaxErr': 1169.824543452351}, 'heston_stats_sw': {'RMSE': 219.45586731562466, 'MAE': 81.59409636127607, 'Bias': -81.58043287397895, 'MaxErr': 1169.8336686367697}, 'vgsa_stats_eq': {'RMSE': 219.4575193730276, 'MAE': 81.59571731274008, 'Bias': -81.57677229303768, 'MaxErr': 1169.834013479352}, 'vgsa_stats_sw': {'RMSE': 219.4596076455114, 'MAE': 81.59743107863859, 'Bias': -81.58143323669141, 'MaxErr': 1169.8398883810703}}
